<a href="https://colab.research.google.com/github/Naveed101633/Retrieval-Augmented-Generation-RAG-Learning-Lab/blob/main/information_retrieval_search_foundation/retrieval_metrics_faiss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 RAG Retrieval Metrics — FAISS Edition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Naveed101633/Retrieval-Augmented-Generation-RAG-Learning-Lab/blob/main/information_retrieval_search_foundation/retrieval_metrics_faiss.ipynb)

**Upgrade from the DeepLearning.AI lab:**
- ✅  → **FAISS IndexFlatIP** (industry standard, ~100x faster)
- ✅  → **Embeddings computed live** in this notebook
- ✅ → **Runs anywhere, zero pre-downloads** (except model)
- ✅ Same `BAAI/bge-base-en-v1.5` model as the original lab
- ✅ Same Precision@K and Recall@K metrics — now with visual plots

**What you'll learn:**
1. How FAISS indexes and searches vector embeddings
2. Why L2-normalizing embeddings converts dot-product to cosine similarity
3. Precision@K vs Recall@K tradeoff with live charts
4. How enterprise RAG systems do retrieval under the hood

---
### Table of Contents
1. [Install & Import](#install)
2. [Load Dataset](#dataset)
3. [Generate Embeddings with BAAI/bge-base-en-v1.5](#embeddings)
4. [Build FAISS Index](#faiss)
5. [Retrieval Function](#retrieval)
6. [Precision@K and Recall@K](#metrics)
7. [Run Queries & Visualize Results](#results)
8. [Key Takeaways](#takeaways)

## 1. Install & Import <a id='install'></a>

> **Runtime tip (Colab):** Go to `Runtime → Change runtime type → T4 GPU` for ~3x faster embedding generation.

In [ ]:
# Install required packages
# faiss-cpu works on both CPU and GPU runtimes in Colab
!pip install -q faiss-cpu sentence-transformers scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import time
import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer
from sklearn.datasets import fetch_20newsgroups

print(f'✅ FAISS version : {faiss.__version__}')
print(f'✅ NumPy version : {np.__version__}')
print(f'✅ All imports successful')

## 2. Load Dataset <a id='dataset'></a>

Same **20 Newsgroups** dataset as the original lab — 11,314 documents across 20 categories.

In [ ]:
# Load the 20 Newsgroups dataset
print('📦 Loading 20 Newsgroups dataset...')
newsgroups_train = fetch_20newsgroups(
    subset='train',
    shuffle=True,
    random_state=42
)

df = pd.DataFrame({
    'text': newsgroups_train.data,
    'category': newsgroups_train.target
})

print(f'✅ Dataset loaded')
print(f'   Documents : {df.shape[0]:,}')
print(f'   Categories: {len(newsgroups_train.target_names)}')
print(f'\n📂 Categories:\n')
for i, name in enumerate(newsgroups_train.target_names):
    count = (df['category'] == i).sum()
    print(f'   [{i:2d}] {name:<35} ({count} docs)')

## 3. Generate Embeddings with BAAI/bge-base-en-v1.5 <a id='embeddings'></a>

**Why this model?**
- `BAAI/bge-base-en-v1.5` is a state-of-the-art open-source embedding model from Beijing Academy of AI
- Produces **768-dimensional** dense vectors
- Ranks among top models on the MTEB (Massive Text Embedding Benchmark) leaderboard
- Used in production by many enterprises as a cost-free alternative to OpenAI embeddings

> ⏱️ **Expected time:** ~4 min on T4 GPU, ~15 min on CPU. The model download is ~438MB (one-time).

In [ ]:
def preprocess_text(text: str) -> str:
    """Strip leading/trailing whitespace from text."""
    return text.strip()

# Load model — downloads from HuggingFace Hub on first run
MODEL_NAME = 'BAAI/bge-base-en-v1.5'
print(f'🤖 Loading model: {MODEL_NAME}')
print('   (First run downloads ~438MB — subsequent runs load from cache)\n')

model = SentenceTransformer(MODEL_NAME)
print(f'✅ Model loaded')
print(f'   Embedding dimension: {model.get_sentence_embedding_dimension()}')

In [ ]:
# Encode all documents
# batch_size=256 maximises GPU utilisation; reduce to 64 if you hit OOM
df_subset = df.head(50).copy()
print(f'⚙️  Encoding {len(df_subset):,} documents...')
t0 = time.time()

texts_clean = [preprocess_text(t) for t in df_subset['text'].tolist()]

embedding_vectors = model.encode(
    texts_clean,
    batch_size=8,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # ← L2-normalize for cosine similarity via dot product
)

elapsed = time.time() - t0
print(f'\n✅ Encoding complete in {elapsed:.1f}s')
print(f'   Matrix shape : {embedding_vectors.shape}  (docs × dims)')
print(f'   Memory usage : {embedding_vectors.nbytes / 1e6:.1f} MB')

# Verify normalization — all norms should be ~1.0
sample_norms = np.linalg.norm(embedding_vectors[:5], axis=1)
print(f'   Sample L2 norms (should be ≈1.0): {sample_norms.round(4)}')

## 4. Build FAISS Index <a id='faiss'></a>

### Why FAISS?

| Approach | 11K docs, K=50 | 1M docs, K=50 |
|---|---|---|
| FAISS IndexFlatIP | ~0.002s | ~0.5s |

**The key insight:** When embeddings are L2-normalized (all norms = 1), **inner product (dot product) = cosine similarity**. FAISS `IndexFlatIP` exploits this with BLAS-optimized matrix multiplication.

```
cosine_sim(a, b) = dot(a, b) / (||a|| * ||b||)
                 = dot(a, b) / (1 * 1)      ← after normalization
                 = dot(a, b)
```

In [ ]:
# Build FAISS index
# IndexFlatIP = exact Inner Product search (no approximation)
# For 1M+ docs you'd use IndexIVFFlat or IndexHNSWFlat for speed

EMBEDDING_DIM = embedding_vectors.shape[1]   # 768

print(f'🏗️  Building FAISS IndexFlatIP (dim={EMBEDDING_DIM})...')

index = faiss.IndexFlatIP(EMBEDDING_DIM)

# FAISS requires float32
vectors_f32 = embedding_vectors.astype(np.float32)
index.add(vectors_f32)

print(f'✅ FAISS index built')
print(f'   Index type   : {type(index).__name__}')
print(f'   Vectors stored: {index.ntotal:,}')

# Quick sanity check — retrieve top-1 for first doc should return itself
test_vec = vectors_f32[0:1]
distances, indices = index.search(test_vec, k=1)
assert indices[0][0] == 0, 'Sanity check failed!'
print(f'   Sanity check : ✅ (doc[0] → top-1 = doc[0], score={distances[0][0]:.4f})')

## 5. Retrieval Function <a id='retrieval'></a>

In [ ]:
def retrieve_documents(query: str, index: faiss.Index, model: SentenceTransformer,
                        df: pd.DataFrame, top_k: int = 5) -> None:
    """
    Retrieve the top-k most relevant documents for a query using FAISS.

    Steps:
      1. Encode and L2-normalize the query
      2. Call index.search() — returns cosine similarity scores + doc indices
      3. Print results

    Args:
        query    : Natural language query string
        index    : Trained FAISS index
        model    : SentenceTransformer model (must match index embeddings)
        df       : DataFrame with 'text' and 'category' columns
        top_k    : Number of documents to retrieve
    """
    # Encode query — normalize so dot product = cosine similarity
    query_embedding = model.encode(
        preprocess_text(query),
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32).reshape(1, -1)   # FAISS expects shape (1, dim)

     # Single FAISS call replaces the entire manual cosine loop
    scores, indices = index.search(query_embedding, k=top_k)

    print(f'🔍 Query: "{query}"')
    print('─' * 70)
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        category = newsgroups_train.target_names[df.iloc[idx]['category']]
        snippet  = df.iloc[idx]['text'][:200].replace('\n', ' ')
        print(f'[{rank}] Score: {score:.4f} | Category: {category}')
        print(f'    {snippet}...')
        print()


# Example
retrieve_documents('space exploration', index, model, df, top_k=3)

## 6. Precision@K and Recall@K <a id='metrics'></a>

### Precision@K
Of the K documents we retrieved, what fraction are actually relevant?
$$\text{Precision@K} = \frac{\text{Relevant docs in top-K}}{K}$$

### Recall@K
Of all relevant documents in the corpus, what fraction did we capture?
$$\text{Recall@K} = \frac{\text{Relevant docs in top-K}}{\text{Total relevant docs in corpus}}$$

**The tradeoff:** Increasing K raises recall (you find more relevant docs) but lowers precision (you also drag in irrelevant ones).

In [ ]:
def precision_at_k(relevant_count: int, k: int) -> float:
    """
    Precision@K = relevant_count / K

    Args:
        relevant_count: Number of relevant docs in top-K results
        k             : Total documents retrieved

    Returns:
        float in [0, 1], or 0.0 if k == 0
    """
    if relevant_count < 0 or k < 0:
        raise ValueError('Inputs must be non-negative.')
    return 0.0 if k == 0 else relevant_count / k


def recall_at_k(relevant_count: int, total_relevant: int) -> float:
    """
    Recall@K = relevant_count / total_relevant

    Args:
        relevant_count : Relevant docs found in top-K
        total_relevant : Total relevant docs in entire corpus

    Returns:
        float in [0, 1], or 0.0 if total_relevant == 0
    """
    if relevant_count < 0 or total_relevant < 0:
        raise ValueError('Inputs must be non-negative.')
    return 0.0 if total_relevant == 0 else relevant_count / total_relevant


print('✅ Metric functions defined')

# Unit tests
assert precision_at_k(4, 5)   == 0.8
assert precision_at_k(0, 5)   == 0.0
assert precision_at_k(5, 0)   == 0.0
assert recall_at_k(5, 100)    == 0.05
assert recall_at_k(0, 100)    == 0.0
assert recall_at_k(5, 0)      == 0.0
print('✅ All unit tests passed')

## 7. Run Queries & Visualize Results <a id='results'></a>

In [ ]:
# Pre-build a lookup: category_name → total docs in corpus (avoids recomputing per query)
category_counts = {
    newsgroups_train.target_names[i]: (df['category'] == i).sum()
    for i in range(len(newsgroups_train.target_names))
}

def compute_metrics(queries: list, index: faiss.Index, model: SentenceTransformer,
                    df: pd.DataFrame, top_k: int = 5) -> list:
    """
    Compute Precision@K and Recall@K for a list of queries using FAISS retrieval.

    Args:
        queries : List of dicts with keys 'query' and 'desired_category'
        index   : FAISS index (must match model embedding dim)
        model   : SentenceTransformer model
        df      : DataFrame with 'category' column
        top_k   : K value for retrieval

    Returns:
        List of result dicts with query, precision@k, recall@k
    """
    results = []

    for item in queries:
        query            = item['query']
        desired_category = item['desired_category']

        # Encode query
        q_emb = model.encode(
            preprocess_text(query),
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype(np.float32).reshape(1, -1)

        # FAISS search — one call, no loop
        _, top_indices = index.search(q_emb, k=top_k)

        # Count relevant docs in top-K
        retrieved_cats     = [newsgroups_train.target_names[df.iloc[idx]['category']]
                               for idx in top_indices[0]]
        relevant_in_top_k  = sum(1 for c in retrieved_cats if c == desired_category)
        total_relevant     = category_counts.get(desired_category, 0)

        results.append({
            'query'       : query,
            'category'    : desired_category,
            'precision@k' : precision_at_k(relevant_in_top_k, top_k),
            'recall@k'    : recall_at_k(relevant_in_top_k, total_relevant),
        })

    return results


print('✅ compute_metrics() defined')

In [ ]:
# Same 10 test queries as the original lab
test_queries = [
    {'query': 'advancements in space exploration technology',        'desired_category': 'sci.space'},
    {'query': 'real-time rendering techniques in computer graphics', 'desired_category': 'comp.graphics'},
    {'query': 'latest findings in cardiovascular medical research',  'desired_category': 'sci.med'},
    {'query': 'NHL playoffs and team performance statistics',        'desired_category': 'rec.sport.hockey'},
    {'query': 'impacts of cryptography in online security',         'desired_category': 'sci.crypt'},
    {'query': 'the role of electronics in modern computing devices', 'desired_category': 'sci.electronics'},
    {'query': 'motorcycles maintenance tips for enthusiasts',        'desired_category': 'rec.motorcycles'},
    {'query': 'high-performance baseball tactics for championships', 'desired_category': 'rec.sport.baseball'},
    {'query': 'historical influence of politics on society',         'desired_category': 'talk.politics.misc'},
    {'query': 'latest technology trends in the Windows operating system', 'desired_category': 'comp.os.ms-windows.misc'},
]

K_VALUES = [5, 20, 50]

# Run all K values and store results
all_results = {}
for k in K_VALUES:
    t0 = time.time()
    all_results[k] = compute_metrics(test_queries, index, model, df, top_k=k)
    elapsed = time.time() - t0
    print(f'K={k:2d} → computed in {elapsed:.3f}s')

print('\n✅ All metrics computed')

In [ ]:
# Print results table (same format as original lab)
for k in K_VALUES:
    print(f"\n{'='*70}")
    print(f'  Results with K={k}')
    print(f"{'='*70}")
    for r in all_results[k]:
        print(f"  Query : {r['query']}")
        print(f"  P@{k:<2} : {r['precision@k']:.2f}   R@{k:<2}: {r['recall@k']:.4f}")
        print()

In [ ]:
# ── Visualization: Precision & Recall across K values ──────────────────────

query_labels = [q['query'][:35] + '...' for q in test_queries]
colors = cm.tab10(np.linspace(0, 1, len(test_queries)))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('RAG Retrieval Metrics — FAISS + BAAI/bge-base-en-v1.5', fontsize=14, fontweight='bold')

# ── Left: Precision@K ──
ax = axes[0]
for i, q in enumerate(test_queries):
    precisions = [all_results[k][i]['precision@k'] for k in K_VALUES]
    ax.plot(K_VALUES, precisions, marker='o', color=colors[i],
            label=query_labels[i], linewidth=2, markersize=7)

ax.set_title('Precision@K  (higher = more relevant results)', fontsize=12)
ax.set_xlabel('K (number of documents retrieved)')
ax.set_ylabel('Precision@K')
ax.set_xticks(K_VALUES)
ax.set_ylim(0, 1.05)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.4, label='Perfect precision')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=7, loc='lower left')

# ── Right: Recall@K ──
ax = axes[1]
for i, q in enumerate(test_queries):
    recalls = [all_results[k][i]['recall@k'] for k in K_VALUES]
    ax.plot(K_VALUES, recalls, marker='o', color=colors[i],
            label=query_labels[i], linewidth=2, markersize=7)

ax.set_title('Recall@K  (higher = more of corpus covered)', fontsize=12)
ax.set_xlabel('K (number of documents retrieved)')
ax.set_ylabel('Recall@K')
ax.set_xticks(K_VALUES)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.savefig('retrieval_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Plot saved as retrieval_metrics.png')

In [ ]:
# ── Average metrics summary ─────────────────────────────────────────────────
print('\n📊 Average Metrics Summary')
print(f'   {"K":<6} {"Avg Precision@K":>16} {"Avg Recall@K":>14}')
print('   ' + '─' * 40)
for k in K_VALUES:
    avg_p = np.mean([r['precision@k'] for r in all_results[k]])
    avg_r = np.mean([r['recall@k']    for r in all_results[k]])
    print(f'   K={k:<4} {avg_p:>16.3f} {avg_r:>14.4f}')

print()
print('💡 Insight: As K increases → Recall ↑ but Precision ↓')
print('   For RAG systems, K=5–20 is typically optimal:')
print('   - High precision (most retrieved docs are relevant)')
print('   - Small context window (cheaper & faster LLM calls)')

## BONUS: FAISS Index Types — When to Use What

This notebook uses `IndexFlatIP` (exact search). Here's the full picture:

| Index Type | Speed | Memory | Accuracy | Use Case |
|---|---|---|---|---|
| `IndexFlatIP` | Slow | High | **100% exact** | < 100K docs, learning, ground truth |
| `IndexIVFFlat` | Fast | Medium | ~98% | 100K–10M docs, most production RAG |
| `IndexHNSWFlat` | Very fast | High | ~99% | Low-latency APIs, real-time search |
| `IndexPQ` | Fastest | **Very low** | ~90% | Billions of vectors, RAM-constrained |

**Most enterprise RAG stacks** (LangChain, LlamaIndex default config) use `IndexFlatIP` or `IndexIVFFlat`.